# AttriBoost AI: NVIDIA cuDF vs. CPU Pandas Performance Profiler

This notebook verifies the multi-touch attribution (MTA) calculation speedups using **NVIDIA cuDF** compared to standard **CPU Pandas**.

### Prerequisite:
Please ensure your Colab runtime is configured with a GPU: **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU**.

In [1]:
!rm -rf AttriBoost-AI
!git clone https://github.com/palakgoda/AttriBoost-AI.git
%cd AttriBoost-AI

Cloning into 'AttriBoost-AI'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 71 (delta 36), reused 55 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 46.42 KiB | 2.44 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/AttriBoost-AI


In [2]:
# 2. Install NVIDIA cuDF compatibility packages for Python 3.10+
!pip install cudf-cu12 --extra-index-url=https://pypi.nvidia.com

Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com


In [3]:
# 3. Run the standalone benchmarking script
import sys
sys.path.append('.')
import os
import pandas as pd
from backend.data_generator import generate_marketing_data
from backend.attribution_engine import run_performance_benchmark, GPU_AVAILABLE

print(f"NVIDIA GPU (cuDF) Available: {GPU_AVAILABLE}")

# Generate standard profiling dataset (200k rows)
os.makedirs("data", exist_ok=True)
generate_marketing_data("data", num_users=200000, total_touchpoints=2000000)

df_tp = pd.read_csv("data/touchpoints.csv")
df_conv = pd.read_csv("data/conversions.csv")

# Run performance comparison
results = run_performance_benchmark(df_tp, df_conv)

print("\n=============================================")
print("           MEASURED BENCHMARK RESULTS         ")
print("=============================================")
for r in results:
    print(f"Scale: {r['dataset_percentage']}% | Rows: {r['rows_processed']:,}")
    print(f"  CPU Pandas: {r['cpu_time_ms']:.2f} ms")
    print(f"  GPU cuDF:   {r['gpu_time_ms']:.2f} ms")
    print(f"  Speedup:    {r['speedup']:.1f}x")
    print(f"  GPU Mode:   {r['gpu_mode']}")
    print("---------------------------------------------")

NVIDIA RAPIDS cuDF is available. GPU acceleration active.
NVIDIA GPU (cuDF) Available: True
Generating data for 200000 users with approx 2000000 touchpoints...
Simulating user journeys...
Converting to DataFrames...
Writing datasets to data...
Successfully generated:
 - Touchpoints: 2013325 rows in data/touchpoints.csv
 - Conversions: 40451 rows in data/conversions.csv

           MEASURED BENCHMARK RESULTS         
Scale: 10% | Rows: 205,377
  CPU Pandas: 102.19 ms
  GPU cuDF:   128.68 ms
  Speedup:    0.8x
  GPU Mode:   NVIDIA cuDF (GPU Active)
---------------------------------------------
Scale: 50% | Rows: 1,026,887
  CPU Pandas: 774.98 ms
  GPU cuDF:   717.79 ms
  Speedup:    1.1x
  GPU Mode:   NVIDIA cuDF (GPU Active)
---------------------------------------------
Scale: 100% | Rows: 2,053,776
  CPU Pandas: 2408.32 ms
  GPU cuDF:   921.28 ms
  Speedup:    2.6x
  GPU Mode:   NVIDIA cuDF (GPU Active)
---------------------------------------------
